# References, Identity, and Copying


```{contents}
:local:
:depth: 2
```


```{index} Missing Is Different from Empty
```



```{index} Missing Is Different from Empty
```

## Missing Is Different from Empty



A reference can identify an object or contain null. An empty order object still exists and can answer Count; a missing order reference cannot receive an instance call. A nullable annotation describes intended use for the compiler's analysis. It is not a runtime constructor check, and adding a null-forgiving exclamation mark does not create an object.

When a relationship is required, reject an absent value at the boundary that establishes the relationship. When it is optional, give absence a deliberate meaning and test that path. Returning a default zero for an absent invoice may confuse missing data with an invoice whose real total is zero. The supporting nullability reading retains the operators and all original variants; use them to implement a policy that has already been stated.

Guided check: distinguish an order with no lines, no order object, and an order whose total is zero after discounts. Which operations are valid on each? A useful object contract keeps these cases distinguishable rather than treating every unusual state as null.

For related explanation, see the further reading below.[^development-reading]


```{index} Objects, References, and State
```

## Objects, References, and State



A manager sees two screens showing the same product. If one screen changes its price, should the other screen change too? The answer depends on whether the screens refer to one shared object or to independent objects. Before combining classes, make that sharing visible.

Chapter 10 taught you to create a valid object. This section asks what happens when you assign that object to another variable, pass it to a method, or store it in a collection. Use a drawing with variables as boxes and objects as separate records. Draw a reference as an arrow, not as another copy of the record.


```{index} Assignment Copies a Reference
```

## Assignment Copies a Reference




```{index} Assignment Copies a Reference
```


A class is a reference type. Assignment copies a reference to an object; it does not call the constructor or copy every field. Two variables that refer to the same object are **aliases**. An alias gives another route to the same state.

Predict both printed prices before running the cell. There is one `StockItem` object and two local variables. The setter checks the new price before replacing the field, so all aliases observe the same valid state.


In [1]:
using System;

StockItem original = new StockItem(10m);
StockItem otherScreen = original;
otherScreen.Price = 12m;
Console.WriteLine($"Original: {original.Price:F2}");
Console.WriteLine($"Other screen: {otherScreen.Price:F2}");
Console.WriteLine($"Same object: {ReferenceEquals(original, otherScreen)}");

class StockItem
{
    private decimal price;
    public StockItem(decimal price) { Price = price; }
    public decimal Price
    {
        get { return price; }
        set
        {
            if (value < 0m) throw new ArgumentOutOfRangeException(nameof(value));
            price = value;
        }
    }
}


Original: 12.00
Other screen: 12.00
Same object: True


The two prices are 12.00. `ReferenceEquals` asks whether two references identify the same object. It does not compare business identifiers or prices. Reassigning `otherScreen` to a new object would change that variable's arrow; it would not redirect `original`.

Passing a reference-type argument to an ordinary parameter copies the reference too. A method can use that reference to change the shared object. Reassigning the parameter itself changes only the method's local copy. This behavior is ordinary pass-by-value: the copied value happens to be a reference. You do not need a `ref` parameter to call a method that changes an object's state.


The drawing the section asked for, before and after the assignment through the alias:

```text
before otherScreen.Price = 12m          after otherScreen.Price = 12m

original    ──▶ ┌──────────────┐        original    ──▶ ┌──────────────┐
                │ StockItem    │                        │ StockItem    │
otherScreen ──▶ │ Price = 10   │        otherScreen ──▶ │ Price = 12   │
                └──────────────┘                        └──────────────┘
```

Two arrows, one record. Compare the independent case, which is what `new` produces each time it runs:

```text
first    ──▶ ┌──────────────┐        imported ──▶ ┌──────────────┐
             │ Product      │                     │ Product      │
             │ Sku = "P100" │                     │ Sku = "P100" │
             └──────────────┘                     └──────────────┘
```

| Situation | Objects | References | `ReferenceEquals` |
| --- | ---: | ---: | --- |
| `otherScreen = original` | 1 | 2 | true |
| two `new Product("P100", ...)` calls | 2 | 2 | false, same SKU |


```{index} Identity and Business Equality
```

## Identity and Business Equality




```{index} Identity and Business Equality
```


**Object identity** means being the same runtime object. **Business equality** means matching according to a stated domain rule. Two independently created product objects may represent the same stock-keeping unit (SKU), even though they have different identities.

For this chapter, SKU comparisons are case-sensitive. The constructor trims surrounding spaces but does not change letter case. Decide that policy before writing comparisons. A title and a price are not reliable identifiers: names can repeat and prices can change across catalog versions.


In [1]:
using System;
using System.Collections.Generic;

Product first = new Product("P100", "Notebook", 6m);
Product imported = new Product("P100", "Notebook", 6m);
Console.WriteLine($"Same object: {ReferenceEquals(first, imported)}");
Console.WriteLine($"Same SKU: {first.Sku == imported.Sku}");

class Product
{
    public string Sku { get; }
    public string Name { get; }
    public decimal UnitPrice { get; }

    public Product(string sku, string name, decimal unitPrice)
    {
        if (string.IsNullOrWhiteSpace(sku)) throw new ArgumentException("SKU required.");
        if (string.IsNullOrWhiteSpace(name)) throw new ArgumentException("Name required.");
        if (unitPrice < 0m) throw new ArgumentOutOfRangeException(nameof(unitPrice));
        Sku = sku.Trim();
        Name = name.Trim();
        UnitPrice = unitPrice;
    }
}


Same object: False
Same SKU: True


The first answer is false and the second is true. This does not mean `==` automatically compares the fields of every class. Here it compares the two `string` SKU values, which have value-based equality. For your own ordinary classes, do not assume that field-by-field equality appears automatically.

Name a method such as `HasSameSku` when that is the rule you need. An application may need several comparisons: identical object, same catalog product, or equal receipt contents. One vague idea of “equal” cannot safely replace all three. Overriding equality and defining operators are optional extensions, not prerequisites for this chapter.


```{index} Mutable and Immutable Objects
```

## Mutable and Immutable Objects




```{index} Mutable and Immutable Objects
```


An object is **mutable** when its observable state can change after construction. An **immutable** object keeps its observable state fixed. The `Product` below has read-only properties and no method that changes its fields. Its property values are strings and a decimal, so sharing it cannot later change the catalog values through a setter.

Read-only access to a reference is not enough to make a whole object immutable. A read-only property could still return a mutable list or another mutable object. You must inspect what callers can reach through the public members.


In [1]:
using System;
using System.Collections.Generic;

Product catalogVersion = new Product("P100", "Notebook", 6m);
Product quote = catalogVersion;
Product revisedCatalog = new Product("P100", "Notebook", 7m);
Console.WriteLine($"Existing quote: {quote.UnitPrice:F2}");
Console.WriteLine($"New catalog: {revisedCatalog.UnitPrice:F2}");

class Product
{
    public string Sku { get; }
    public string Name { get; }
    public decimal UnitPrice { get; }

    public Product(string sku, string name, decimal unitPrice)
    {
        if (string.IsNullOrWhiteSpace(sku)) throw new ArgumentException("SKU required.");
        if (string.IsNullOrWhiteSpace(name)) throw new ArgumentException("Name required.");
        if (unitPrice < 0m) throw new ArgumentOutOfRangeException(nameof(unitPrice));
        Sku = sku.Trim();
        Name = name.Trim();
        UnitPrice = unitPrice;
    }
}


Existing quote: 6.00
New catalog: 7.00


Changing the catalog means creating another `Product`; the existing quote still refers to the original version. This is one simple policy for preserving quoted prices. A larger sales system might instead store an explicit sale-time price on each order line. Either approach needs a stated rule for which price belongs to an existing order.

Use mutability when changes are part of the object's job, such as adding lines to an order. Prefer fixed value objects when sharing should be harmless, such as a product description used by several order lines. Immutability is a design choice, not a requirement that every class avoid setters.


```{index} Copies Need a Depth Policy
```

## Copies Need a Depth Policy




```{index} Copies Need a Depth Policy
```


Creating a new outer object does not automatically duplicate everything it references. A **shallow copy** copies the immediate values and references. A **deep copy** also copies selected referenced objects. Deep copying is a policy about a graph of objects; it is not a universal command to duplicate everything.

Consider two orders that both refer to one immutable `Product`. Independent order collections with a shared product can be correct. If the product were mutable, a price edit could affect both orders. Draw the reachable objects and identify which changes the design permits before deciding that a copy is “independent.”

The next section uses fixed products and order lines, together with an order that owns its changing collection. That makes the sharing policy small enough to explain and test.


```{index} Practice: Draw the Arrows
```

## Practice: Draw the Arrows




```{index} Practice: Draw the Arrows
```


Create an alias and an independent customer copy. Rename through the alias. Print both names and whether the alias identifies the original object. Explain your drawing before running.


In [ ]:
using System;

Customer first = new Customer("North Office");
Customer alias = first; // Explain why this shares an object.
Customer independent = new Customer(first.Name);
// Your code starts here.
/* TODO: rename through alias. */
// Your code ends here.
Console.WriteLine(first.Name);
Console.WriteLine(independent.Name);
Console.WriteLine(ReferenceEquals(first, alias));

class Customer
{
    public string Name { get; set; }
    public Customer(string name) { Name = name; }
}


In [1]:
using System;

Customer first = new Customer("North Office");
Customer alias = first;
Customer independent = new Customer(first.Name);
alias.Name = "Central Office";
Console.WriteLine(first.Name);
Console.WriteLine(independent.Name);
Console.WriteLine(ReferenceEquals(first, alias));

class Customer
{
    public string Name { get; set; }
    public Customer(string name) { Name = name; }
}


Central Office
North Office
True


```{rubric} Footnotes
```


[^1101_objects_references-1]: A variable that stores a reference is separate from the object it refers to. Reference diagrams describe observable relationships; they do not promise a particular physical memory address or garbage-collector layout.
[^1101_objects_references-structs]: Microsoft, [Structure types](https://learn.microsoft.com/en-us/dotnet/csharp/language-reference/builtin-types/struct). See value copying, readonly structures, and default values.
[^1101_objects_references-records]: Microsoft, [Records](https://learn.microsoft.com/en-us/dotnet/csharp/language-reference/builtin-types/record). See positional properties, equality, and shallow copying with `with`.


[^development-reading]: Further reading: [Microsoft, C# Record Types](https://learn.microsoft.com/en-us/dotnet/csharp/fundamentals/types/records). The worked business scenarios and guided reasoning here are original teaching examples.
